In [1]:
import pandas as pd
import numpy as np
import pyaging as pya

In [2]:
path = "D:/bioTest/attack/"
pheno = pd.read_excel(f"{path}controls.xlsx", index_col=0)
betas = pd.read_pickle(f"{path}betas.pkl")

feats_pheno = ['Age', 'Sex', 'Tissue']
pheno = pheno[feats_pheno]

df_clocks = pd.merge(pheno, betas, left_index=True, right_index=True)
df_clocks['Female'] = (df_clocks['Sex'] == 'F').astype(int)

In [3]:
clocks = ["horvath2013"]

adata = pya.pp.df_to_adata(df_clocks, metadata_cols=['Sex', 'Tissue'], imputer_strategy='knn', verbose=True)
pya.pred.predict_age(adata=adata, dir=path, clock_names=clocks, verbose=True)
results = pd.merge(pheno.loc[:, feats_pheno], adata.obs[clocks], left_index=True, right_index=True)
results.to_excel(f"{path}/clock.xlsx")

|-----> 🏗️ Starting df_to_adata function
|-----> ⚙️ Create anndata object started
|-----> ✅ Create anndata object finished [1.1624s]
|-----> ⚙️ Add metadata to anndata started
|-----------> Adding provided metadata to adata.obs
|-----> ✅ Add metadata to anndata finished [0.0020s]
|-----> ⚙️ Log data statistics started
|-----------> There are 729 observations
|-----------> There are 411976 features
|-----------> Total missing values: 0
|-----------> Percentage of missing values: 0.00%
|-----> ✅ Log data statistics finished [0.3684s]
|-----> ⚙️ Impute missing values started
|-----------> No missing values found. No imputation necessary
|-----> ✅ Impute missing values finished [0.3695s]
|-----> 🎉 Done! [2.4030s]
|-----> 🏗️ Starting predict_age function
|-----> ⚙️ Set PyTorch device started
|-----------> Using device: cpu
|-----> ✅ Set PyTorch device finished [0.0016s]
|-----> 🕒 Processing clock: horvath2013
|-----------> ⚙️ Load clock started
|-----------------> Downloading data to D:/bio

In [4]:
logger = pya.logger.Logger('test_logger')
device = 'cpu'
dir = 'pyaging_data'
indent_level = 1

clock = pya.pred.load_clock(clocks[0], device, dir, logger, indent_level=indent_level)
clock_features = list(clock.features)

|-----> ⚙️ Load clock started
|-----------> Downloading data to pyaging_data\horvath2013.pt
|-----------> in progress: 100.0000%
|-----> ✅ Load clock finished [0.5492s]


In [7]:
common_cpgs = list(set(clock_features).intersection(betas.columns))
differ_cpgs = list(set(clock_features).difference(betas.columns))
df = pd.merge(betas.loc[:, common_cpgs], results, left_index=True, right_index=True)

In [ ]:
df = pd.read_excel(f"{path}/data/immuno/models/SImAge/data.xlsx", index_col='sample_id')
feats = pd.read_excel(f"{path}/data/immuno/models/SImAge/feats_con_top10.xlsx", index_col=0).index.values
ids_feat = list(range(len(feats)))
col_trgt = 'Age'
col_pred = 'SImAge'

df_preds = pd.read_excel(f"{path}/data/immuno/models/SImAge/results/predictions.xlsx", index_col=0)
ids_trn = df_preds.index[df_preds['fold_0002'] == 'trn'].values
ids_val = df_preds.index[df_preds['fold_0002'] == 'val'].values
ids_tst = df_preds.index[df_preds['fold_0002'] == 'tst_ctrl_central'].values
ids_all = df_preds.index[df_preds['fold_0002'].isin(['trn', 'val', 'tst_ctrl_central'])].values
ids_trn_val = df_preds.index[df_preds['fold_0002'].isin(['trn', 'val'])].values
ids_dict = {
    'all': ids_all,
    'trn_val': ids_trn_val,
    'tst': ids_tst
}

df = df.loc[ids_all, :]
df["SImAge Error"] = df["SImAge"] - df["Age"]
df["|SImAge Error|"] = df["SImAge Error"].abs()
df['Data'] = 'Real'
df['Eps'] = 'Origin'

model = WDFTTransformerModel.load_from_checkpoint(checkpoint_path=f"{path}/data/immuno/models/SImAge/best_fold_0002.ckpt")
model.eval()
model.freeze()

def predict_func_regression(X):
    model.produce_probabilities = True
    batch = {
        'all': torch.from_numpy(np.float32(X[:, ids_feat])),
        'continuous': torch.from_numpy(np.float32(X[:, ids_feat])),
        'categorical': torch.from_numpy(np.int32(X[:, []])),
    }
    tmp = model(batch)
    return tmp.cpu().detach().numpy()

art_regressor = PyTorchRegressor(
    model=model,
    loss=model.loss_fn,
    input_shape=[len(feats)],
    optimizer=torch.optim.Adam(
        params=model.parameters(),
        lr=model.hparams.optimizer_lr,
        weight_decay=model.hparams.optimizer_weight_decay
    ),
    use_amp=False,
    opt_level="O1",
    loss_scale="dynamic",
    channels_first=True,
    clip_values=None,
    preprocessing_defences=None,
    postprocessing_defences=None,
    preprocessing=(0.0, 1.0),
    device_type="cpu",
)

colors_atks = {
    "MomentumIterative": px.colors.qualitative.D3[0],
    "BasicIterative": px.colors.qualitative.D3[1],
    "FastGradient": px.colors.qualitative.D3[3],
}

df.to_excel(f"{path_save}/df_origin.xlsx", index_label='sample_id')